# Fine-tune Llama 3.2 1B for Vietnamese Medical QA

This notebook fine-tunes `unsloth/Llama-3.2-1B-Instruct-bnb-4bit` on `hungnm/vietnamese-medical-qa` with Unsloth LoRA adapters, `TrainingArguments`, and early stopping so the resulting adapter can be served by this repo's `medqa-lora` deployment flow.

In [ ]:
%pip install torch==2.10.0 --index-url https://download.pytorch.org/whl/cu130
%pip install unsloth trl datasets huggingface_hub peft

In [1]:
import torch
torch.__version__

'2.10.0+cu130'

In [2]:
import os
from pathlib import Path
import torch
import unsloth
from datasets import DatasetDict, load_dataset
from huggingface_hub import login
from transformers import DataCollatorForLanguageModeling, EarlyStoppingCallback, TrainingArguments, set_seed
from trl.trainer.sft_trainer import SFTTrainer
from unsloth import FastLanguageModel

set_seed(42)

MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
DATASET_NAME = "hungnm/vietnamese-medical-qa"
WORKDIR = Path.cwd().resolve()
OUTPUT_DIR = WORKDIR / "artifacts" / "medqa-lora-llama32-1b"
MAX_SEQ_LENGTH = 2048
TRAIN_TEST_SPLIT = 0.05
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_THRESHOLD = 0.0
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
SYSTEM_PROMPT = (
    "Bạn là trợ lý AI hỗ trợ thông tin y khoa bằng tiếng Việt. "
    "Trả lời rõ ràng, thận trọng, không chẩn đoán quá mức, "
    "và khuyên người dùng gặp bác sĩ khi có dấu hiệu nguy hiểm."
)

HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"BF16 available: {USE_BF16}")
print(f"Unsloth version: {getattr(unsloth, '__version__', 'unknown')}")
print(f"Artifacts directory: {OUTPUT_DIR}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


Device: cuda
BF16 available: True
Unsloth version: 2025.11.1
Artifacts directory: /home/thuannd/Repository/aio-llmops/notebooks/artifacts/medqa-lora-llama32-1b


In [3]:
raw_dataset: DatasetDict = load_dataset(DATASET_NAME)
split_dataset = raw_dataset["train"].train_test_split(test_size=TRAIN_TEST_SPLIT, seed=42)
dataset = DatasetDict({
    "train": split_dataset["train"],
    "val": split_dataset["test"],
})

print(dataset)
sample = dataset["train"][0]
print("\nQuestion preview:\n")
print(sample["question"][:500])
print("\nAnswer preview:\n")
print(sample["answer"][:500])

DatasetDict({
    train: Dataset({
        features: ['answer', 'question'],
        num_rows: 8868
    })
    val: Dataset({
        features: ['answer', 'question'],
        num_rows: 467
    })
})

Question preview:

Xin chào bác sĩ, thông thường tôi rất ít khi ốm đau nhưng vài năm trở lại đây mỗi lần ốm phải nằm chỗ, tôi có các biểu hiện sau: Rét run người, đầu đau như búa bổ, sốt cao rồi vã mồ hôi. Xin hỏi bác sĩ đó có phải là triệu chứng của bệnh sốt rét không ạ. Nếu là sốt rét tôi phải điều trị như thế nào? Xin bác sĩ nói rõ hơn về bệnh sốt rét để tôi có phương hướng điều trị kịp thời ạ!
Lê Hoàng Anh (Hà Nội)
Chào bác sĩ. Bố tôi bị sốt và lạnh có phải là bị sốt rét không ạ? Mong bác sĩ tư vấn giúp ạ. Cảm

Answer preview:

Chào bạn, sốt rét là bệnh truyền nhiễm gây ra do muỗi đốt, nhất là ở khu vực miền núi. Triệu chứng sốt rét gồm: Lạnh run, sốt, nhức đầu, vã mồ hôi, mệt mỏi và tùy trường hợp có thể có triệu chứng nặng hơn: Co giật, vàng da, suy hô hấp. Để chẩn đoán sốt rét thì 

In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
 )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "up_proj",
        "down_proj",
        "o_proj",
        "gate_proj",
    ],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state=42,
 )
model.config.use_cache = False
model.print_trainable_parameters()

==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.555 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.11.1 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [5]:
def build_messages(question, answer=None):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question.strip()},
    ]
    if answer is not None:
        messages.append({"role": "assistant", "content": answer.strip()})
    return messages

def render_conversation(question, answer=None, add_generation_prompt=False):
    messages = build_messages(question, answer)
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

    prompt = (
        f"{SYSTEM_PROMPT}\n\n"
        f"Câu hỏi:\n{question.strip()}\n\n"
        "Trả lời:\n"
    )
    if answer is None:
        return prompt
    return f"{prompt}{answer.strip()}{tokenizer.eos_token}"

def format_batch(batch):
    texts = [
        render_conversation(question, answer)
        for question, answer in zip(batch["question"], batch["answer"])
    ]
    return {"text": texts}

formatted_dataset = dataset.map(
    format_batch,
    batched=True,
    remove_columns=dataset["train"].column_names,
 )

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH)

tokenized_dataset = formatted_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
 )

print(formatted_dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 8868
    })
    val: Dataset({
        features: ['text'],
        num_rows: 467
    })
})


In [6]:
print(render_conversation(dataset["train"][10]["question"], dataset["train"][10]["answer"]))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 23 Apr 2026

Bạn là trợ lý AI hỗ trợ thông tin y khoa bằng tiếng Việt. Trả lời rõ ràng, thận trọng, không chẩn đoán quá mức, và khuyên người dùng gặp bác sĩ khi có dấu hiệu nguy hiểm.<|eot_id|><|start_header_id|>user<|end_header_id|>

Xin chào bác sĩ
Thời gian gần đây tôi hay bị tê chân nhất là các đầu ngón chân . Không biết tôi bị bệnh gì , dùng thuốc nào để chữa<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Chào bạn 
Bị tê chân và đặc biệt là các đầu ngón chân là do thiếu máu đến các đầu chi, cũng có thể là do bị thiếu Canxi hoặc thiếu Vitamin nhóm B. Vì vậy : 
- Hàng ngày nên ngâm chân tay vào nước ấm từ 10-15p. xoa bóp các đầu chi và uống thuốc : 
+ Piracetam 400mg x 4v/ngày/ 2 lần ( 30 ngày ) 
+ Vitamin 3B x 3v/ngày / 2 lần ( 30 ngày ) 
+ Canxi D3 x 2v/ngày / 2 lần ( 30 ngày ) 
- Tập thể dục thường xuyên hàng ngày nhẹ nhàng , không ngồi quá lâu và tăng cường vận

In [7]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=200,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    lr_scheduler_type="cosine",
    report_to="none",
    bf16=USE_BF16,
    fp16=torch.cuda.is_available() and not USE_BF16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    seed=42,
 )

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["val"],
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
 )

train_result = trainer.train()
train_result.metrics

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,868 | Num Epochs = 5 | Total steps = 2,775
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,2.003900,1.916556
200,1.818000,1.783298
300,1.746600,1.722063
400,1.694100,1.685681
500,1.708700,1.651149
600,1.529100,1.636745
700,1.529200,1.619779
800,1.503000,1.603114
900,1.584600,1.590720
1000,1.469400,1.579090


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


{'train_runtime': 2010.2547,
 'train_samples_per_second': 22.057,
 'train_steps_per_second': 1.38,
 'total_flos': 7.29902385489838e+16,
 'train_loss': 1.6566513054187482,
 'epoch': 2.3426510369702434}

In [8]:
eval_metrics = trainer.evaluate()
eval_metrics

{'eval_loss': 1.5956823825836182,
 'eval_runtime': 23.5986,
 'eval_samples_per_second': 19.789,
 'eval_steps_per_second': 2.5,
 'epoch': 2.3426510369702434}

In [9]:
FastLanguageModel.for_inference(model)

question = dataset["val"][0]["question"]
reference_answer = dataset["val"][0]["answer"]
prompt = render_conversation(question, answer=None, add_generation_prompt=True)
model_device = next(model.parameters()).device
inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

with torch.inference_mode():
    generated = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=1.1,
    )

generated_tokens = generated[0][inputs["input_ids"].shape[1]:]
prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("Question:\n")
print(question)
print("\nModel answer:\n")
print(prediction)
print("\nReference answer:\n")
print(reference_answer)

Question:

Xin chào bác sĩ,

Gần đây cháu hay bị đau đầu khi làm việc nhiều, vùng bị đau là vùng sau gáy bên phải.
Thông thường làm việc tập trung liên tục khoảng 2-3h thì sẽ đau và không làm việc tiếp đc. Nếu nghỉ tầm 15-20' thì sẽ đỡ đau hơn.

Do đặc thù công việc cháu chưa đi khám đc. Mong đc bác sĩ tư vấn ạ

Model answer:

Chào bạn,
Đau đầu có rất nhiều nguyên nhân, có thể do căng thẳng, stress, thiếu ngủ, rối loạn giấc ngủ, bệnh lý mạch máu não, bệnh lý thần kinh, hoặc các bệnh lý toàn thân khác...
Để xác định được tình trạng của bạn, bạn nên đến chuyên khoa Nội Thần Kinh để kiểm tra thêm nhé.
Thân mến!

Reference answer:

Chào bạn,
Hiện nay tình trạng thoái hóa xương khớp rất phổ biến ở nhân viên văn phòng do ngồi một chỗ , không vận động nhiều, hay làm việc trước máy tính, ít rèn luyện cơ thể và thường xuyên bị nặng ở những người làm việc lao động chân tay, những người cao tuổi.
Bạn nên hạn chế ngồi lâu, nên tập thể dục giữa giờ các bài tập cổ, vai , gáy có thể tập tại chỗ từ 15

In [10]:
final_adapter_dir = OUTPUT_DIR / "final_adapter"
final_adapter_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)

print(f"Saved LoRA adapter to: {final_adapter_dir.resolve()}")
print("Use this directory when loading the adapter into vLLM as medqa-lora.")

Saved LoRA adapter to: /home/thuannd/Repository/aio-llmops/notebooks/artifacts/medqa-lora-llama32-1b/final_adapter
Use this directory when loading the adapter into vLLM as medqa-lora.


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

ckpt_path = "./artifacts/medqa-lora-llama32-1b/final_adapter"
tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
model = AutoModelForCausalLM.from_pretrained(ckpt_path)

In [12]:
model.push_to_hub(
    "thuanan/Llama-3.2-1B-Instruct-vi-medqa-lora",
    commit_message="Add model ckpt",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/thuanan/Llama-3.2-1B-Instruct-vi-medqa-lora/commit/f6a0a884ab160b023b4f649216a3b0f595d99064', commit_message='Add model ckpt', commit_description='', oid='f6a0a884ab160b023b4f649216a3b0f595d99064', pr_url=None, repo_url=RepoUrl('https://huggingface.co/thuanan/Llama-3.2-1B-Instruct-vi-medqa-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='thuanan/Llama-3.2-1B-Instruct-vi-medqa-lora'), pr_revision=None, pr_num=None)

In [13]:
tokenizer.push_to_hub(
    "thuanan/Llama-3.2-1B-Instruct-vi-medqa-lora",
    commit_message="Add tokenizer ckpt",
)

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/thuanan/Llama-3.2-1B-Instruct-vi-medqa-lora/commit/1e08fb51aa0a1411ee0da175f1630967d8d38c64', commit_message='Add tokenizer ckpt', commit_description='', oid='1e08fb51aa0a1411ee0da175f1630967d8d38c64', pr_url=None, repo_url=RepoUrl('https://huggingface.co/thuanan/Llama-3.2-1B-Instruct-vi-medqa-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='thuanan/Llama-3.2-1B-Instruct-vi-medqa-lora'), pr_revision=None, pr_num=None)